# 07. Spatial Extremes

This notebook uses `RMC.BestFit.Models.SpatialExtremes.SpatialGEV` directly with intercept-only trend functions for location, scale, and shape.

In [4]:
import pythonnet
pythonnet.load("coreclr")

import clr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from helper_functions import resolve_bestfit_dll, resolve_numerics_dll, convert_to_dotnet_array

clr.AddReference(str(resolve_numerics_dll()))
clr.AddReference(str(resolve_bestfit_dll()))

from System import Array, Double
from RMC.BestFit.Models.TrendFunctions import GeneralLinearFunction
from RMC.BestFit.Models.SpatialExtremes import SpatialGEV

def to_dotnet_2d(a):
    a = np.asarray(a, dtype=float)
    arr = Array.CreateInstance(Double, a.shape[0], a.shape[1])
    for i in range(a.shape[0]):
        for j in range(a.shape[1]):
            arr[i, j] = float(a[i, j])
    return arr

In [ ]:
rng = np.random.default_rng(25)
coords = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, 0.5]])
index_flood = np.array([8000, 9000, 11000, 10000, 12000], dtype=float)
n_years = 30

# At-site annual maxima matrix: rows=years, columns=sites.
at_site = np.vstack([
    index_flood * rng.lognormal(0, 0.18, size=len(index_flood))
    for _ in range(n_years)
])

# Intercept-only linear functions keep the first demo focused on the SpatialGEV API.
X = np.ones((coords.shape[0], 1), dtype=float)
loc_fun = GeneralLinearFunction('Location', to_dotnet_2d(X))
sca_fun = GeneralLinearFunction('Scale', to_dotnet_2d(X))
shp_fun = GeneralLinearFunction('Shape', to_dotnet_2d(X))

model = SpatialGEV(to_dotnet_2d(at_site), to_dotnet_2d(coords), loc_fun, sca_fun, shp_fun)
model.SetDefaultParameters()
p0 = np.array([float(p.Value) for p in model.Parameters], dtype=float)
ll = float(model.LogLikelihood(convert_to_dotnet_array(p0)))

summary = pd.DataFrame([
    {
        'model': 'SpatialGEV',
        'log_likelihood_at_default': ll,
        'n_parameters': int(model.NumberOfParameters),
        'n_sites': int(coords.shape[0]),
        'n_years': int(n_years),
    }
])
summary


In [ ]:
# Derive a quick representative Q100 from the current default marginal parameters.
model.SetParameterValues(convert_to_dotnet_array(p0))

# Read current linear-function intercepts from model component parameters.
loc0 = float(model.Location.Parameters[0].Value)
sca0 = float(model.Scale.Parameters[0].Value)
shp0 = float(model.Shape.Parameters[0].Value)

# Approximate a representative at-site Q100 from the marginal GEV component.
from Numerics.Distributions import GeneralizedExtremeValue
gev = GeneralizedExtremeValue(loc0, max(1e-6, abs(sca0)), shp0)
q100 = float(gev.InverseCDF(0.99))

print('SpatialGEV default-parameter log-likelihood:', round(ll, 3))
print('Marginal parameter snapshot (location, scale, shape):', (round(loc0, 3), round(sca0, 3), round(shp0, 3)))
print('Representative marginal Q100:', round(q100, 2))


In [ ]:
plt.figure(figsize=(8,5))
for s in range(at_site.shape[1]):
    plt.scatter(np.repeat(s+1, at_site.shape[0]), at_site[:, s], s=12, alpha=0.35)
plt.xticks(range(1, at_site.shape[1]+1))
plt.xlabel('Site index')
plt.ylabel('Annual peak')
plt.title('Spatial Extremes Input Data (Site-wise)')
plt.grid(alpha=0.25)
plt.tight_layout()

## Summary and Exercises

You used BestFit `SpatialGEV` directly with trend functions and evaluated the default-parameter log-likelihood.

Recommended exercises:
1. Add covariates (elevation, area) to `GeneralLinearFunction` instead of intercept-only.
2. Expand to more sites and test sensitivity to coordinate scaling.
3. Build Q100 maps using site-specific marginals from fitted component functions.